# Learning Notes

Personal log of what I've learned working on this project: concepts, tools, gotchas, things I want to remember.

## What is YAML

YAML ("YAML Ain't Markup Language") is a human-readable format for structured data, mostly used for config files. It uses indentation instead of brackets/braces, `key: value` pairs, and `- ` for list items.

The `data.yaml` file the Roboflow dataset download produced is a real example:

```yaml
path: C:/Users/rohan/Desktop/Quant Sports Project/experiments/Football-Players-Detection-1
train: train/images
val: valid/images
test: test/images

nc: 4
names: ['ball', 'goalkeeper', 'player', 'referee']
```

- `path`, `train`, `val`, `test`: tell YOLO where to find the dataset's image folders.
- `nc`: number of classes (4 here).
- `names`: the class labels, in the same order the model will predict them.

Nesting works by indentation, for example the `roboflow:` block underneath that has its own `workspace`, `project`, `version`, etc. as sub-keys. No special syntax needed, just consistent indentation.


## COCO weights, and why fine-tuning beats starting from scratch

COCO (Common Objects in Context) is a large general-purpose dataset, about 330,000 images across 80 everyday object classes (person, car, dog, sports ball, tv, etc.). "COCO-pretrained weights" means the model has already been trained on that dataset, so it already knows general visual concepts: edges, shapes, textures, what a person-shaped blob looks like from different angles, and so on.

That's exactly what `yolo26n.pt` was in the first smoke test: a COCO-pretrained checkpoint, used purely for inference (no training). It worked reasonably for the generic `person` class but was unreliable for `sports ball` and had no concept of soccer-specific roles (player vs. referee vs. goalkeeper), because COCO never taught it those distinctions.

**Fine-tuning** (what the Roboflow retrain decision is doing) means continuing training from those COCO-pretrained weights on a new, task-specific dataset, instead of starting from random weights. This is a form of transfer learning:

- The early layers of the network already know general visual features, so training doesn't have to relearn "what an edge looks like" from zero.
- Only the later layers really need to adapt, mainly the classification head, to the new classes (`ball`, `goalkeeper`, `player`, `referee`).
- This means far less data and far less compute is needed to get a good result, compared to training an object detector completely from scratch, which typically needs hundreds of thousands of images.

This is why the plan is "fine-tune YOLO26 on the Roboflow dataset" rather than "train a brand new detector from nothing."


## Why 4GB VRAM is little, and what CUDA actually does

**VRAM** is the memory that lives on the GPU itself (separate from regular system RAM). During training, VRAM has to hold: the model's weights, the batch of images currently being processed, all the intermediate activations from the forward pass, and the gradients from the backward pass, all at once. All of that competes for the same pool of memory.

4GB is small by current standards. The GTX 1650 in this machine is a budget/laptop-class GPU; modern training GPUs (RTX 4090, A100, H100) have 24GB to 80GB+. The practical consequence: if the batch size or image size is too large for the available VRAM, training crashes with an out-of-memory (OOM) error rather than just running slower. That's why the training run started with `batch=16` instead of YOLO's larger default, as a conservative starting point, with room to raise it if VRAM allows or lower it if it OOMs.

**CUDA** is NVIDIA's platform that lets software run computations directly on the GPU instead of the CPU. The reason this matters for deep learning: a CPU has a small number of powerful, general-purpose cores, while a GPU has thousands of simpler cores built to do the same operation on lots of data at once. Deep learning is mostly large matrix multiplications, which is exactly the kind of highly-parallel, repetitive math GPUs are built for. CUDA is the layer that lets PyTorch hand those matrix operations off to the GPU's cores instead of the CPU's.

Concretely, in this project: `torch` was first installed as a CPU-only build (`2.14.0+cpu`), so `torch.cuda.is_available()` returned `False` and training would have run on the CPU, slowly. Reinstalling `torch`/`torchvision` from the CUDA-enabled index (`cu130`, matching this machine's driver) fixed that; `torch.cuda.is_available()` now returns `True` and the GTX 1650 is detected and usable for training.


## Lesson (2026-09-18): local GPU training on old/underpowered hardware is a real hardware risk, not just slow

While attempting to run training locally on the GTX 1650, a capacitor on the machine blew, likely a short circuit related to the power adapter under sustained load. Fixed, no lasting damage, but the practical lesson is:

Training deep learning models pushes a machine's power delivery and cooling much harder than normal use, for a sustained period (not a quick spike). On older or already-marginal hardware, that sustained load is a genuine risk to the hardware itself, not just something that runs slowly.

**Decision going forward:** use an external/cloud GPU (Google Colab, or similar) for actual training runs instead of pushing local hardware. Local setup is still fine for quick inference smoke tests (like the original YOLO detection test), just not for sustained training workloads on this machine.
